In [ ]:
####Remasking and aligning National variables
# -*- coding: utf-8 -*-
import os
import time
import hashlib
import arcpy

from arcpy.sa import ExtractByMask

# ============================================================
# USER SETTINGS
# ============================================================

INPUT_FOLDER = r"C:\Users\acosta_pedro\OneDrive - Norges geologiske undersøkelse\Geochemistry NGU_2025\kalk_prosjekt3.0\National\Variables_National_250m\To_resample_mask_10m"

MASK_RASTER = r"E:\Alpha_earth\qc\alphaearth_mosaic_epsg25833_band1_qc_cog.tif"

# WRITE FINAL OUTPUTS LOCALLY, NOT TO ONEDRIVE
OUTPUT_FOLDER = r"D:\arc_outputs_resampled_10m"

OVERWRITE_EXISTING = False

# Continuous rasters: BILINEAR or CUBIC
# Categorical rasters: NEAREST
RESAMPLING_METHOD = "BILINEAR"

VALID_EXTENSIONS = (".tif", ".tiff", ".img", ".bil", ".asc")

# ASCII-safe temporary workspace
TEMP_ROOT = r"C:\arc_tmp_resample"
TEMP_GDB = os.path.join(TEMP_ROOT, "temp.gdb")
TEMP_STAGE = os.path.join(TEMP_ROOT, "stage")

# If True, keep a normal TIFF when COG creation fails
KEEP_TIFF_IF_COG_FAILS = True

# ============================================================
# HELPERS
# ============================================================

def ts():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def msg(s):
    text = f"[{ts()}] {s}"
    try:
        arcpy.AddMessage(text)
    except Exception:
        pass
    print(text, flush=True)

def safe_delete(path):
    try:
        if arcpy.Exists(path):
            arcpy.management.Delete(path)
        elif os.path.exists(path):
            os.remove(path)
    except Exception:
        pass

def output_exists_for_skip(cog_path, tif_path):
    if OVERWRITE_EXISTING:
        return False
    return os.path.exists(cog_path) or os.path.exists(tif_path)

def list_input_rasters(folder):
    rasters = []
    for fn in os.listdir(folder):
        full = os.path.join(folder, fn)
        if os.path.isfile(full) and fn.lower().endswith(VALID_EXTENSIONS):
            rasters.append(full)
    return sorted(rasters)

def get_base_name(path):
    return os.path.splitext(os.path.basename(path))[0]

def short_safe_id(name):
    # Very short, ASCII-safe ID for temp datasets
    h = hashlib.md5(name.encode("utf-8")).hexdigest()[:8]
    return f"r_{h}"

def ensure_temp_workspace():
    os.makedirs(TEMP_ROOT, exist_ok=True)
    os.makedirs(TEMP_STAGE, exist_ok=True)
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    if not arcpy.Exists(TEMP_GDB):
        arcpy.management.CreateFileGDB(TEMP_ROOT, "temp.gdb")

def report_raster_info(label, raster_path):
    if not arcpy.Exists(raster_path):
        msg(f"{label}: not found")
        return

    r = arcpy.Raster(raster_path)
    e = r.extent
    msg(f"{label}:")
    msg(f"  rows/cols   = {r.height} x {r.width}")
    msg(f"  cell size   = {r.meanCellWidth} x {r.meanCellHeight}")
    msg(f"  extent      = XMin={e.XMin}, YMin={e.YMin}, XMax={e.XMax}, YMax={e.YMax}")

def compare_to_mask(mask_path, out_path, tol=1e-9):
    if not arcpy.Exists(mask_path) or not arcpy.Exists(out_path):
        return False

    m = arcpy.Raster(mask_path)
    r = arcpy.Raster(out_path)

    me = m.extent
    re = r.extent

    same_rows = (m.height == r.height)
    same_cols = (m.width == r.width)
    same_cx = abs(m.meanCellWidth - r.meanCellWidth) < tol
    same_cy = abs(m.meanCellHeight - r.meanCellHeight) < tol
    same_xmin = abs(me.XMin - re.XMin) < tol
    same_ymin = abs(me.YMin - re.YMin) < tol
    same_xmax = abs(me.XMax - re.XMax) < tol
    same_ymax = abs(me.YMax - re.YMax) < tol

    ok = all([same_rows, same_cols, same_cx, same_cy, same_xmin, same_ymin, same_xmax, same_ymax])

    if ok:
        msg(f"{os.path.basename(out_path)} matches MASK exactly.")
    else:
        msg(f"{os.path.basename(out_path)} does NOT match MASK exactly.")
        msg(f"  rows match:   {same_rows}")
        msg(f"  cols match:   {same_cols}")
        msg(f"  cell x match: {same_cx}")
        msg(f"  cell y match: {same_cy}")
        msg(f"  xmin match:   {same_xmin}")
        msg(f"  ymin match:   {same_ymin}")
        msg(f"  xmax match:   {same_xmax}")
        msg(f"  ymax match:   {same_ymax}")

    return ok

def write_stage_tiff(in_raster, out_tif, mask_raster, mask_sr):
    safe_delete(out_tif)
    with arcpy.EnvManager(
        snapRaster=mask_raster,
        cellSize=mask_raster,
        extent=mask_raster,
        outputCoordinateSystem=mask_sr
    ):
        arcpy.management.CopyRaster(
            in_raster=in_raster,
            out_rasterdataset=out_tif,
            format="TIFF"
        )

def convert_tiff_to_cog(in_tif, out_cog, mask_raster, mask_sr):
    safe_delete(out_cog)
    with arcpy.EnvManager(
        snapRaster=mask_raster,
        cellSize=mask_raster,
        extent=mask_raster,
        outputCoordinateSystem=mask_sr
    ):
        arcpy.management.CopyRaster(
            in_raster=in_tif,
            out_rasterdataset=out_cog,
            format="COG"
        )

# ============================================================
# CORE PROCESSING
# ============================================================

def process_one_raster(in_raster, mask_raster, output_folder, mask_sr, mask_cellsize,
                       resampling_method, scratch_gdb):
    base = get_base_name(in_raster)
    rid = short_safe_id(base)

    out_cog = os.path.join(output_folder, f"{base}_10m_cog.tif")
    out_tif = os.path.join(output_folder, f"{base}_10m.tif")

    if output_exists_for_skip(out_cog, out_tif):
        msg(f"Skipping existing output for: {base}")
        return

    tmp_proj = os.path.join(scratch_gdb, f"{rid}_p")
    tmp_mask = os.path.join(scratch_gdb, f"{rid}_m")
    stage_tif = os.path.join(TEMP_STAGE, f"{rid}_stage.tif")

    safe_delete(tmp_proj)
    safe_delete(tmp_mask)
    safe_delete(stage_tif)

    msg("------------------------------------------------------------")
    msg(f"START: {os.path.basename(in_raster)}")

    try:
        msg("Projecting / resampling to MASK grid...")
        with arcpy.EnvManager(
            snapRaster=mask_raster,
            cellSize=mask_raster,
            extent=mask_raster,
            outputCoordinateSystem=mask_sr
        ):
            arcpy.management.ProjectRaster(
                in_raster=in_raster,
                out_raster=tmp_proj,
                out_coor_system=mask_sr,
                resampling_type=resampling_method,
                cell_size=mask_cellsize
            )

        msg("Applying MASK...")
        with arcpy.EnvManager(
            snapRaster=mask_raster,
            cellSize=mask_raster,
            extent=mask_raster,
            outputCoordinateSystem=mask_sr
        ):
            ExtractByMask(tmp_proj, mask_raster).save(tmp_mask)

        msg(f"Writing staged TIFF: {stage_tif}")
        write_stage_tiff(tmp_mask, stage_tif, mask_raster, mask_sr)

        msg(f"Creating final COG: {out_cog}")
        try:
            convert_tiff_to_cog(stage_tif, out_cog, mask_raster, mask_sr)
            compare_to_mask(mask_raster, out_cog)

            # If COG succeeded, staged TIFF can be removed
            safe_delete(stage_tif)

        except Exception as cog_err:
            msg("COG creation failed.")
            msg(str(cog_err))

            if KEEP_TIFF_IF_COG_FAILS:
                msg(f"Keeping regular TIFF instead: {out_tif}")
                safe_delete(out_tif)
                arcpy.management.CopyRaster(
                    in_raster=stage_tif,
                    out_rasterdataset=out_tif,
                    format="TIFF"
                )
                compare_to_mask(mask_raster, out_tif)
            else:
                raise

        msg(f"END: {os.path.basename(in_raster)}")

    except Exception as e:
        msg(f"FAILED: {os.path.basename(in_raster)}")
        msg(str(e))
        raise

    finally:
        safe_delete(tmp_proj)
        safe_delete(tmp_mask)
        msg("------------------------------------------------------------")

# ============================================================
# MAIN
# ============================================================

def main():
    t0 = time.time()

    arcpy.env.overwriteOutput = True
    arcpy.CheckOutExtension("Spatial")

    ensure_temp_workspace()

    mask_desc = arcpy.Describe(MASK_RASTER)
    mask_sr = mask_desc.spatialReference
    mask_ras = arcpy.Raster(MASK_RASTER)
    mask_cellsize = mask_ras.meanCellWidth

    if abs(mask_ras.meanCellWidth - mask_ras.meanCellHeight) > 1e-9:
        raise ValueError("MASK raster does not have square cells.")

    arcpy.env.snapRaster = MASK_RASTER
    arcpy.env.cellSize = MASK_RASTER
    arcpy.env.extent = MASK_RASTER
    arcpy.env.outputCoordinateSystem = mask_sr

    msg("MASK reference:")
    report_raster_info("MASK", MASK_RASTER)

    rasters = list_input_rasters(INPUT_FOLDER)

    if len(rasters) == 0:
        msg("No input rasters found.")
        return

    msg(f"Found {len(rasters)} rasters to process.")
    msg(f"Resampling method: {RESAMPLING_METHOD}")
    msg(f"Target cell size: {mask_cellsize}")
    msg(f"Output folder: {OUTPUT_FOLDER}")
    msg(f"Resume mode: {'SKIP existing outputs' if not OVERWRITE_EXISTING else 'OVERWRITE outputs'}")
    msg(f"Keep TIFF if COG fails: {KEEP_TIFF_IF_COG_FAILS}")

    scratch_gdb = TEMP_GDB

    for r in rasters:
        process_one_raster(
            in_raster=r,
            mask_raster=MASK_RASTER,
            output_folder=OUTPUT_FOLDER,
            mask_sr=mask_sr,
            mask_cellsize=mask_cellsize,
            resampling_method=RESAMPLING_METHOD,
            scratch_gdb=scratch_gdb
        )

    msg("Finished all rasters.")
    msg(f"Total time: {(time.time() - t0) / 60:.1f} minutes")

if __name__ == "__main__":
    main()